# 🎓 Predicting Student Mental Health Score from Social Media Usage

**Problem:** We're predicting a student's `Mental_Health_Score` (a continuous score from ~3 to ~10) using their social media habits, study time, sleep, physical activity, and stress level. This is a **regression** problem since the target is a number, not a category.

**Dataset:** `Student_Social_Media_And_Mental_Health_Impact.csv` — 5,000 students, 13 columns covering demographics (Age, Gender, Country), platform usage (Avg_Daily_Usage_Hours, Daily_Unlocks, Most_Used_Platform), lifestyle (Study_Hours, Sleep_Hours_Per_Night, Physical_Activity_Hours), and Stress_Level.



## 1. Import Libraries

We're grouping imports by purpose: data handling, visualization, preprocessing, models, evaluation, and saving. Keeping imports grouped like this makes it easy to see at a glance what the notebook is going to do.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load the Dataset

We load the CSV and take a first look at its shape and the first few rows. This is always the very first step — you can't make any decisions about a dataset until you've actually looked at it.

In [ ]:
df = pd.read_csv('Student Social Media And Mental Health Impact.csv')

In [ ]:
df.describe()

In [ ]:
  #Looking for number of rows and columns.
df.shape

In [ ]:
#Look at first 5 rows.
df.head()

📌 **Data quality:** this dataset contains two duplicate rows and no missing values. We will remove duplicates before splitting so the same record cannot appear in both train and test sets. The target is a dataset score, not a clinical diagnosis.


In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

📌 **Data quality:** `Physical_Activity_Hours` includes a minimum of -0.4 hours, an impossible value. We cap negative activity at zero during cleaning while retaining the rest of the row.


## 4. Exploratory Data Analysis (EDA)

We'll keep this focused — six plots, each answering one specific question about the data, not a wall of charts for the sake of it.

### 4.1 — Distribution of the target (`Mental_Health_Score`)

Before predicting anything, we need to understand the shape of what we're predicting.

In [ ]:
sns.histplot(df['Mental_Health_Score'], kde=True)

### 4.2 — Correlation heatmap

Which numeric features actually move together with the target?

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

### 4.3 — Stress Level vs Mental Health Score

Does higher stress genuinely come with a lower score?

In [ ]:
df.columns

In [ ]:
df['Stress_Level'].unique()

In [ ]:
order = ['Low', 'Medium', 'High', 'Very High']
sns.boxplot(x='Stress_Level', y='Mental_Health_Score',data=df, order=order)

### 4.4 — Daily Usage Hours vs Mental Health Score

Does more time on social media relate to a lower score?

In [ ]:
sns.scatterplot(x='Avg_Daily_Usage_Hours', y='Mental_Health_Score',data=df)

### 4.5 — Sleep Hours vs Mental Health Score

Sleep is one of the most commonly cited mental health factors — does the data back that up?

In [ ]:
df.columns

In [ ]:
sns.scatterplot(x='Sleep_Hours_Per_Night', y='Mental_Health_Score', data=df)

### 4.6 — Most Used Platform (count)

A quick look at which platforms are most common in our dataset.

In [ ]:
df.columns

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x=df['Most_Used_Platform'], order=df['Most_Used_Platform'].value_counts().index)

**Checking Outliers**

In [ ]:
num_features = df.select_dtypes(include='number') #int64 or float64
Q1  = num_features.quantile(0.25)
Q3  = num_features.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (num_features < lower_bound) | (num_features > upper_bound)
print(outliers.sum())

## 5. Data Cleaning

Remove duplicate records and correct the single impossible negative physical-activity value. The target and remaining feature values are retained.


In [ ]:
df.describe()

In [ ]:
# Remove repeated records, then correct impossible negative activity values.
df = df.drop_duplicates().copy()
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)
print(f'Rows after cleaning: {len(df):,}')
print(f'Remaining duplicates: {df.duplicated().sum()}')
print(f'Minimum physical activity: {df["Physical_Activity_Hours"].min()}')


Clipping only the impossible negative value preserves the other useful information in that student's row. Duplicate removal happens before splitting. No rows are removed for ordinary high or low values.


## 6. Skewness

**What skewness is:** a measure of how lopsided a numeric column's distribution is. A value near 0 means roughly symmetric (bell-shaped); a large positive or negative value means the data leans heavily to one side, with a long tail.

**Why it matters:** models like Linear Regression assume features are roughly well-behaved. A heavily skewed column (think: a long tail of extreme values) can quietly drag the model's predictions in that direction. Tree-based models like Random Forest don't care about skew — but since we're also training a Linear Regression baseline, it's worth fixing.

In [ ]:
num_cols = df.select_dtypes(include='number')
num_cols.skew()
# near to 0 -> Centralized (0.01, 0.002)
# negative -> Left Skewed (-1.56)
# poistive (greater than 0) -> Right Skewed (1.256)

## 7. Feature Engineering

`Country` has many distinct values. We retain its information and group infrequent countries using frequency counts from the **training set only**. Applying that mapping to the test set avoids letting test data influence preprocessing.


In [ ]:
# The country grouping is learned from X_train only in the train/test split section.


In [ ]:
# Unseen or infrequent countries map to 'Other' at prediction time.


In [ ]:
# See the split and training-only mapping in section 9.


In [ ]:
# Country categories are prepared after splitting to prevent leakage.


The category mapping is fit from training data and reused unchanged for the test set and future predictions.


## 8. Encoding Strategy

`Stress_Level` has an explicit natural order (Low, Medium, High, Very High), so it uses ordinal encoding. Gender, academic level, platform, purpose, and grouped country use one-hot encoding. Numeric fields are imputed and scaled in pipelines fit only on training data.


In [ ]:
stress_order = ['Low', 'Medium', 'High', 'Very High']
print('Stress encoding:', {level: code for code, level in enumerate(stress_order)})


## 9. Train-Test Split

**Why we split:** we need data the model has never seen to honestly check how well it generalizes. If we evaluate on the same data we trained on, the model can just "memorize" the answers and look artificially good.

**Common ratios:** 80/20 or 70/30 are both standard. We'll use an 80/20 split here — with 5,000 rows, that still leaves 1,000 rows for a solid, reliable test.

We split **before** fitting any preprocessing — this is the single most common beginner mistake (called data leakage), and we'll avoid it by fitting everything only on the training set.

In [ ]:
from sklearn.model_selection import train_test_split

# Split raw rows before learning category frequencies or fitting transformations.
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

top_countries = train_df['Country'].value_counts().head(10).index.tolist()
def group_country(series):
    return series.where(series.isin(top_countries), 'Other')

feature_cols = [
    'Study_Hours', 'Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
    'Physical_Activity_Hours', 'Sleep_Hours_Per_Night', 'Stress_Level',
    'Gender', 'Academic_Level', 'Most_Used_Platform', 'Purpose_Of_Use', 'Country'
]
skewed_cols = ['Study_Hours']
other_numeric_cols = ['Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
                      'Physical_Activity_Hours', 'Sleep_Hours_Per_Night']
ordinal_cols = ['Stress_Level']
nominal_cols = ['Gender', 'Academic_Level', 'Most_Used_Platform',
                'Purpose_Of_Use', 'Grouped_country']

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
X_train['Grouped_country'] = group_country(X_train.pop('Country'))
X_test['Grouped_country'] = group_country(X_test.pop('Country'))
y_train = train_df['Mental_Health_Score']
y_test = test_df['Mental_Health_Score']
print(f'Train rows: {len(X_train):,}; test rows: {len(X_test):,}')
print(f'Top countries learned from training data: {top_countries}')


## 10. Preprocessing using ColumnTransformer

Our columns need **different treatment**:

- `Study_Hours` (the skewed one) → impute → `log1p` transform → scale
- The other numeric columns → impute → scale (no skew to fix)
- `Stress_Level` → impute → `OrdinalEncoder` with an **explicit order**
- The nominal columns → impute → `OneHotEncoder`

We include a `SimpleImputer` in every branch even though this dataset has zero missing values right now — it's a safety net. Real-world data (and our future API's incoming requests) won't always be this clean, and a pipeline that assumes "no missing values ever" is a pipeline that breaks in production.

`ColumnTransformer` glues all of this into **one object** that applies the right transformation to the right column type in a single `.fit()` / `.transform()` call — no manual column-by-column juggling.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Every learned transformation lives inside the pipeline to prevent leakage.
skew_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('log_transform', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler())
])
plain_numeric_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
ordinal_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=[stress_order], handle_unknown='use_encoded_value', unknown_value=-1))
])
nominal_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=5))
])
preprocessor = ColumnTransformer(transformers=[
    ('skewed', skew_pipeline, skewed_cols),
    ('numeric', plain_numeric_pipeline, other_numeric_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('nominal', nominal_pipeline, nominal_cols)
])


## 11. Build a Pipeline

**Why Pipeline matters:** a `Pipeline` chains preprocessing and the model into a single object. Calling `.fit()` once does both steps in the correct order, and calling `.predict()` on brand-new raw data automatically applies the exact same preprocessing that was used during training — no risk of forgetting a step or applying it inconsistently.

**Why companies prefer this:** when this model gets deployed (which we're doing in Part 2 with FastAPI), the API doesn't need to know *anything* about scaling, encoding, or log transforms — it just loads one saved pipeline object and calls `.predict()` on raw input. That's a huge reduction in what can go wrong in production.

We'll build two pipelines below — one per model — so we can fairly compare them.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42, n_jobs=-1))
])


## 12. Model Building

We're comparing **two models** here because the comparison itself teaches something useful — a plain Linear Regression baseline versus a Random Forest, which can capture the non-linear patterns we spotted back in EDA (section 4.4).

### 12.1 — Baseline: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds       = lr_pipeline.predict(X_test)
lr_preds_train = lr_pipeline.predict(X_train)

lr_r2_testing  = r2_score(y_test, lr_preds)
lr_r2_training = r2_score(y_train, lr_preds_train)
lr_mae         = mean_absolute_error(y_test, lr_preds)

print(f"Accuracy of Training {lr_r2_training}")
print(f"Accuracy of Testing {lr_r2_testing}")
print(f'MAE {lr_mae}')

### 12.2 — Random Forest (default settings)

Same preprocessor, different model at the end — that's the entire point of building the pipeline once.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds          = rf_pipeline.predict(X_test)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing  = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

### 12.3 — Hyperparameter Tuning the Random Forest

When tuning a `Pipeline`, each parameter needs to be prefixed with the step name (`regressor__`) so `RandomizedSearchCV` knows which part of the pipeline it belongs to. We use `RandomizedSearchCV` instead of a full grid search — it samples a fixed number of combinations instead of testing every single one, which is far faster while still finding a strong set of parameters.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'random forest__n_estimators': [100, 200, 300],
    'random forest__max_depth': [5, 10, 15, None],
    'random forest__min_samples_split': [2, 5, 10],
    'random forest__min_samples_leaf': [1, 2, 4]
}
random_search = RandomizedSearchCV(
    estimator=rf_pipeline, param_distributions=param_grid, n_iter=12, cv=3,
    scoring='r2', random_state=42, n_jobs=-1, refit=True
)
random_search.fit(X_train, y_train)


In [ ]:
random_search.best_params_

In [ ]:
rf_best_pipeline = random_search.best_estimator_
rf_best_preds    = rf_best_pipeline.predict(X_test)

print(f"R2 Score for random forest after hyperparameter tuning {r2_score(y_test, rf_best_preds)}")
print(f"MAE Score for random forest after hyperparameter tuning {mean_absolute_error(y_test, rf_best_preds)}")

## 13. Model Evaluation

We report R², MAE, and RMSE on the held-out test set. R² measures explained variation; MAE gives the average absolute error in score points; RMSE penalizes larger errors more. These metrics describe prediction on this dataset and should not be interpreted as clinical assessment.


In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds          = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse           = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training       = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae            = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2             = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)


In [ ]:
# Compare held-out predictions with observed scores and inspect residuals.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, rf_tuned_preds, alpha=0.55)
lo, hi = min(y_test.min(), rf_tuned_preds.min()), max(y_test.max(), rf_tuned_preds.max())
axes[0].plot([lo, hi], [lo, hi], '--', color='black')
axes[0].set(title='Tuned Random Forest: predicted vs actual', xlabel='Actual score', ylabel='Predicted score')
residuals = y_test - rf_tuned_preds
axes[1].scatter(rf_tuned_preds, residuals, alpha=0.55)
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(title='Residuals', xlabel='Predicted score', ylabel='Actual - predicted')
plt.tight_layout()


## 14. Save the model for the API

The API accepts the original input fields and applies the same training-only country mapping and fitted preprocessing. After reviewing test metrics, run the training script to create `artifacts/mental_health_model.joblib`. See `README.md` for setup and launch instructions.
